# Testing the MC-PILCO Functionality

In [6]:
# %load ~/dev/marthaler/header.py
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
# Enable Float64 for more stable matrix inversions.
import jax
import equinox as eqx
from jax import Array, config
import jax.numpy as jnp
import numpy as np
import jax.random as jr
from jaxtyping import ArrayLike, install_import_hook, Array, Float, Int, PyTree  
from typing import Optional
import matplotlib as mpl
import matplotlib.pyplot as plt

config.update("jax_enable_x64", True)

cols = mpl.rcParams["axes.prop_cycle"].by_key()["color"]

In [9]:
from jax_mc_pilco.controllers import Controller, RandomController, SumOfGaussians
from jax_mc_pilco.rewards import pendulum_cost#, cart_pole_cost
from jax_mc_pilco.model_learning.gp_models import IMGPR
from jax_mc_pilco.policy_learning.rollout import fit_controller
from jax_mc_pilco.simulators.simulation import remake_state, sample_from_environment

In [10]:
import optax as ox

In [11]:
from IPython import display

## Globals

In [16]:
key = jr.key(42)

# Test Rollout

In [20]:
def rollout(
    policy: eqx.Module,
    init_samples: ArrayLike,
    model: eqx.Module,
    timesteps: ArrayLike,
    key: ArrayLike = jr.key(123),
) -> Float:
    policy_params, policy_static = eqx.partition(policy, eqx.is_array)

    def one_rollout_step(
        carry: Tuple[ArrayLike, ArrayLike, ArrayLike, Float], 
        timestep: Float
    ) -> Tuple[Tuple[ArrayLike, ArrayLike, ArrayLike, Float], Float]:
        policy_params, key, samples, total_cost = carry
        policy = eqx.combine(policy_params, policy_static)
        actions = jax.vmap(policy)(samples, jnp.tile(timestep, num_particles))

        key, subkey = jr.split(key)
        samples = model.get_samples(key, samples, actions, num_samples=1)
        cost = jnp.mean(jax.vmap(obj_func)(jnp.hstack((samples, actions))))
        return (policy_params, key, samples, total_cost + cost), cost

    total_cost = 0
    (policy_params, key, samples, total_cost), result = jax.lax.scan(
        one_rollout_step, (policy_params, key, init_samples, total_cost), timesteps
    )
    return total_cost

In [24]:
def test_policy(
    states: ArrayLike,
    timestep: Optional[Float] = None,
    key: Optional[ArrayLike] = None
)->Array:
    return jnp.ones_like(states)
    

In [25]:
class TestModel(eqx.Module):
    def get_samples(
        self,
        key: ArrayLike,
        samples: ArrayLike,
        actions: ArrayLike,
        num_samples: Int,
    )->Array:
        return samples
        

In [ ]:
def test_obj(
    
)